In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install scikit-bio

In [3]:
import json
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from mpl_toolkits.mplot3d import Axes3D

from scipy import stats
from scipy.stats import ttest_ind
from scipy.stats import mannwhitneyu
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import pdist, squareform

from skbio.stats.distance import DistanceMatrix, permanova

from statsmodels.multivariate.manova import MANOVA
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.mixture import GaussianMixture
from sklearn.cluster import AgglomerativeClustering, KMeans, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score

from sklearn.ensemble import IsolationForest

from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import roc_curve, roc_auc_score, silhouette_score, r2_score, mean_squared_error
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [4]:
df = pd.read_excel("train_data.xlsx")

In [5]:
df

,Permeability,LDL_uptake,Total_ROS,Vascular_Marker,Cell_Signalling,Tube_formation,In_vivo_recovery,Group
0,20,1.20,1.0,100,80,100,70,1
1,23,1.10,0.9,98,78,98,78,1
2,22,1.40,1.1,96,85,99,79,1
3,27,1.20,1.2,90,89,90,83,1
4,21,1.80,0.8,92,95,96,82,1
...,...,...,...,...,...,...,...,...
79,85,0.32,2.5,75,47,44,31,0
80,77,0.35,2.2,61,41,32,33,0
81,74,0.27,2.1,67,42,32,32,0
82,67,0.60,2.3,66,46,37,30,0


In [6]:
# Shuffle original dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [7]:
# Standardization
def standardization(x):
    scaler = StandardScaler()
    x_scaled = scaler.fit_transform(x)
    return x_scaled, scaler

In [8]:
def predict_train_summary(model_path):

    # Load model artifact
    data = joblib.load(model_path)
    train_summary = data["train_summary"]

    # Return both
    return {
        "train_summary": train_summary
    }

In [9]:
predict_train_summary("/content/drive/MyDrive/RepoGo_Rasp/models/train_data_summary.pkl")

{'train_summary': [{'Group': 0,
   'Permeability_count': 42.0,
   'Permeability_mean': 72.07142857142857,
   'Permeability_std': 7.664911681170199,
   'Permeability_min': 56.0,
   'Permeability_25%': 67.0,
   'Permeability_50%': 72.0,
   'Permeability_75%': 77.0,
   'Permeability_max': 90.0,
   'LDL_uptake_count': 42.0,
   'LDL_uptake_mean': 0.4107142857142857,
   'LDL_uptake_std': 0.45054651289156106,
   'LDL_uptake_min': 0.1,
   'LDL_uptake_25%': 0.2,
   'LDL_uptake_50%': 0.3,
   'LDL_uptake_75%': 0.35,
   'LDL_uptake_max': 2.0,
   'Total_ROS_count': 42.0,
   'Total_ROS_mean': 2.0309523809523813,
   'Total_ROS_std': 0.33458244471470105,
   'Total_ROS_min': 0.9,
   'Total_ROS_25%': 1.9,
   'Total_ROS_50%': 2.1,
   'Total_ROS_75%': 2.275,
   'Total_ROS_max': 2.6,
   'Vascular_Marker_count': 42.0,
   'Vascular_Marker_mean': 69.04761904761905,
   'Vascular_Marker_std': 5.547807217676763,
   'Vascular_Marker_min': 59.0,
   'Vascular_Marker_25%': 65.0,
   'Vascular_Marker_50%': 68.5,
   'V

# Statistical Modelling

In [10]:
def statistics(df):
    summary_dict = df.describe().to_dict()
    return json.dumps(summary_dict, indent=4)

In [11]:
# Group Wise Descriptive Statistics for each feature
def group_wise_stats(df, label_col):
    if label_col not in df.columns:
      raise ValueError(f"{label_col} column not found in dataframe.")

    result = {}

    for col in df.columns[:-1]:
        stats = df.groupby(label_col)[col].describe().round(2)
        result[col] = stats.to_dict()

    return json.dumps(result, indent=4)

## t-test

In [ ]:
def t_test(df, label_col):

    if label_col not in df.columns:
        raise ValueError(f"{label_col} column not found in dataframe.")

    # Select numeric features except label
    features = df.select_dtypes(include=[np.number]).columns.drop(label_col)

    # Get unique groups
    groups = df[label_col].unique()

    if len(groups) != 2:
        raise ValueError("Independent t-test requires exactly two groups.")

    g1_label, g0_label = groups

    results = {}

    for feature in features:

        # Extract actual data for each group
        group1 = df[df[label_col] == g1_label][feature].dropna()
        group0 = df[df[label_col] == g0_label][feature].dropna()

        if len(group1) < 2 or len(group0) < 2:
            continue

        t_stat, p_value = ttest_ind(group1, group0, equal_var=False)

        pooled_std = np.sqrt((group1.var() + group0.var()) / 2)

        cohens_d = 0.0 if pooled_std == 0 else \
            (group1.mean() - group0.mean()) / pooled_std

        results[feature] = {
            "group1_label": g1_label,
            "group0_label": g0_label,
            "mean_group1": float(round(group1.mean(), 3)),
            "mean_group0": float(round(group0.mean(), 3)),
            "t_statistic": float(round(t_stat, 3)),
            "p_value": float(round(p_value, 6)),
            "cohens_d": float(round(cohens_d, 3)),
            "significant": bool(p_value < 0.05)
        }

    return results

## Mann–Whitney U Test

In [ ]:
def mann_whitney_test(df, label_col):

    if label_col not in df.columns:
        raise ValueError(f"Label column '{label_col}' not found.")

    if df[label_col].nunique() != 2:
        raise ValueError("Mann–Whitney test requires exactly 2 groups.")

    results = {}

    # Numeric columns except label
    numeric_cols = df.select_dtypes(include=[np.number]).columns.drop(label_col)

    for feature in numeric_cols:

        group_values = df[[feature, label_col]].dropna()

        groups = group_values[label_col].unique()
        g0 = group_values[group_values[label_col] == groups[0]][feature]
        g1 = group_values[group_values[label_col] == groups[1]][feature]

        if len(g0) < 1 or len(g1) < 1:
            continue

        try:
            u_stat, p_val = mannwhitneyu(g0, g1, alternative='two-sided')
        except Exception:
            continue

        results[feature] = {
            "median_group1": float(round(g0.median(), 3)),
            "median_group2": float(round(g1.median(), 3)),
            "U_statistic": float(round(u_stat, 3)),
            "p_value": float(round(p_val, 6)),
            "significant": bool(p_val < 0.05)
        }

    return results

## MANOVA Test

In [ ]:
def manova(df, label_col):

    if label_col not in df.columns:
        raise ValueError(f"{label_col} not found in dataframe.")

    if df[label_col].nunique() < 2:
        raise ValueError("MANOVA requires at least two groups.")

    # Select numeric features except label
    numeric_cols = df.select_dtypes(include=[np.number]).columns.drop(label_col)

    if len(numeric_cols) < 1:
        raise ValueError("No numeric dependent variables found.")

    # Build formula dynamically
    dependent_vars = " + ".join(numeric_cols)
    formula = f"{dependent_vars} ~ {label_col}"

    try:
        manova = MANOVA.from_formula(formula, data=df)
        result = manova.mv_test()
    except Exception as e:
        raise RuntimeError(f"MANOVA failed: {str(e)}")

    output = {
        "method": "MANOVA",
        "effects": {}
    }

    for effect, effect_data in result.results.items():
        stat_table = effect_data["stat"]

        effect_dict = {}

        for test_name, row in stat_table.iterrows():

            # Normalize test name
            normalized_name = (
                test_name.lower()
                .replace(" ", "_")
                .replace("-", "_")
                .replace("'", "")
            )

            effect_dict[normalized_name] = {
                "Value": float(row["Value"]),
                "Num DF": float(row["Num DF"]),
                "Den DF": float(row["Den DF"]),
                "F Value": float(row["F Value"]),
                "Pr > F": float(row["Pr > F"])
            }

        output["effects"][effect] = effect_dict

    return output

## PERMANOVA Test

In [ ]:
def permanova_test(df, label_col, metric="euclidean", permutations=999):

    # dataframe validation
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if df.empty:
        raise ValueError("Dataset is empty.")

    if label_col not in df.columns:
        raise ValueError(f"Label column '{label_col}' not found.")

    # group validation
    if df[label_col].nunique() < 2:
        raise ValueError("PERMANOVA requires at least two groups.")

    # automatically select numeric features
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col in feature_cols:
        feature_cols.remove(label_col)

    if len(feature_cols) == 0:
        raise ValueError("No numeric features found for PERMANOVA.")

    # missing value check
    cols = feature_cols + [label_col]
    if df[cols].isnull().any().any():
        raise ValueError("Missing values detected. Please clean the dataset.")

    # feature matrix
    X = df[feature_cols].values
    groups = df[label_col].values

    try:
        dist_matrix = squareform(pdist(X, metric=metric))
        dm = DistanceMatrix(dist_matrix)

        result = permanova(
            distance_matrix=dm,
            grouping=groups,
            permutations=permutations
        )

    except Exception as e:
        raise RuntimeError(f"PERMANOVA failed: {str(e)}")

    return {
        "method_name": "PERMANOVA",
        "metric": metric,
        "permutations": permutations,
        "test_statistic_name": "pseudo-F",
        "test_statistic": float(result["test statistic"]),
        "p_value": float(result["p-value"]),
        "sample_size": int(result["sample size"]),
        "number_of_groups": int(result["number of groups"]),
        "significant": bool(result["p-value"] < 0.05)
    }

## Correlation Heatmap

In [ ]:
def correlation_matrix(df, label_col=None, method="pearson"):

    if method not in ["pearson","spearman","kendall"]:
        raise ValueError("Method must be 'pearson', 'spearman', or 'kendall'.")

    # dataframe validation
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if df.empty:
        raise ValueError("Dataset is empty.")

    # select numeric features
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    # remove label column if numeric
    if label_col is not None and label_col in feature_cols:
        feature_cols.remove(label_col)

    if len(feature_cols) < 2:
        raise ValueError("At least two numeric features are required to compute correlation.")

    # missing value check
    if df[feature_cols].isnull().any().any():
        raise ValueError("Missing values detected in dataset.")

    # compute correlation matrix
    corr_matrix = df[feature_cols].corr(method=method).round(3)

    return {
        "method_used": method,
        "features": feature_cols,
        "correlation_matrix": corr_matrix.to_dict()
    }

In [ ]:
df.head()

,Permeability,LDL_uptake,Total_ROS,Vascular_Marker,Cell_Signalling,Tube_formation,In_vivo_recovery,Group
0,74,0.30,2.2,61,47,31,33,0
1,20,1.20,1.0,100,80,100,70,1
2,73,0.20,1.7,67,45,44,32,0
3,22,1.20,0.8,96,79,92,72,1
4,79,0.19,1.8,70,43,41,28,0


In [ ]:
df_label0 = df[df['Group'] == 0]

In [ ]:
df_label1 = df[df['Group'] == 1]

In [ ]:
df_label0.info()

<class 'pandas.core.frame.DataFrame'>
Index: 42 entries, 0 to 83
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Permeability      42 non-null     int64  
 1   LDL_uptake        42 non-null     float64
 2   Total_ROS         42 non-null     float64
 3   Vascular_Marker   42 non-null     int64  
 4   Cell_Signalling   42 non-null     int64  
 5   Tube_formation    42 non-null     int64  
 6   In_vivo_recovery  42 non-null     int64  
 7   Group             42 non-null     int64  
dtypes: float64(2), int64(6)
memory usage: 3.0 KB


In [ ]:
correlation_matrix(df_label0, label_col="Group", method="pearson")

{'method_used': 'pearson',
 'features': ['Permeability',
  'LDL_uptake',
  'Total_ROS',
  'Vascular_Marker',
  'Cell_Signalling',
  'Tube_formation',
  'In_vivo_recovery'],
 'correlation_matrix': {'Permeability': {'Permeability': 1.0,
   'LDL_uptake': 0.269,
   'Total_ROS': 0.08,
   'Vascular_Marker': 0.194,
   'Cell_Signalling': 0.003,
   'Tube_formation': -0.055,
   'In_vivo_recovery': 0.168},
  'LDL_uptake': {'Permeability': 0.269,
   'LDL_uptake': 1.0,
   'Total_ROS': 0.056,
   'Vascular_Marker': 0.105,
   'Cell_Signalling': -0.017,
   'Tube_formation': -0.085,
   'In_vivo_recovery': 0.061},
  'Total_ROS': {'Permeability': 0.08,
   'LDL_uptake': 0.056,
   'Total_ROS': 1.0,
   'Vascular_Marker': -0.149,
   'Cell_Signalling': -0.233,
   'Tube_formation': -0.136,
   'In_vivo_recovery': 0.163},
  'Vascular_Marker': {'Permeability': 0.194,
   'LDL_uptake': 0.105,
   'Total_ROS': -0.149,
   'Vascular_Marker': 1.0,
   'Cell_Signalling': 0.503,
   'Tube_formation': 0.11,
   'In_vivo_recove

In [ ]:
df_label1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 42 entries, 1 to 82
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Permeability      42 non-null     int64  
 1   LDL_uptake        42 non-null     float64
 2   Total_ROS         42 non-null     float64
 3   Vascular_Marker   42 non-null     int64  
 4   Cell_Signalling   42 non-null     int64  
 5   Tube_formation    42 non-null     int64  
 6   In_vivo_recovery  42 non-null     int64  
 7   Group             42 non-null     int64  
dtypes: float64(2), int64(6)
memory usage: 3.0 KB


In [ ]:
correlation_matrix(df_label1, label_col="Group", method="pearson")

{'method_used': 'pearson',
 'features': ['Permeability',
  'LDL_uptake',
  'Total_ROS',
  'Vascular_Marker',
  'Cell_Signalling',
  'Tube_formation',
  'In_vivo_recovery'],
 'correlation_matrix': {'Permeability': {'Permeability': 1.0,
   'LDL_uptake': -0.051,
   'Total_ROS': 0.393,
   'Vascular_Marker': 0.123,
   'Cell_Signalling': -0.2,
   'Tube_formation': -0.251,
   'In_vivo_recovery': 0.533},
  'LDL_uptake': {'Permeability': -0.051,
   'LDL_uptake': 1.0,
   'Total_ROS': -0.341,
   'Vascular_Marker': -0.14,
   'Cell_Signalling': 0.497,
   'Tube_formation': 0.007,
   'In_vivo_recovery': 0.138},
  'Total_ROS': {'Permeability': 0.393,
   'LDL_uptake': -0.341,
   'Total_ROS': 1.0,
   'Vascular_Marker': 0.355,
   'Cell_Signalling': -0.32,
   'Tube_formation': -0.07,
   'In_vivo_recovery': 0.322},
  'Vascular_Marker': {'Permeability': 0.123,
   'LDL_uptake': -0.14,
   'Total_ROS': 0.355,
   'Vascular_Marker': 1.0,
   'Cell_Signalling': -0.022,
   'Tube_formation': 0.011,
   'In_vivo_recov

In [ ]:
def statistical_modelliing(df, test_name, label_col=None, **kwargs):
    if test_name == "df_summary":
        return statistics(df)

    elif test_name == "group_wise_summary":
        return group_wise_stats(df, label_col)

    elif test_name == "t_test":
        return t_test(df, label_col)

    elif test_name == "mann_whitney_test":
        return mann_whitney_test(df, label_col)

    elif test_name == "manova":
        return manova(df, label_col)

    elif test_name == "permanova":
        return permanova_test(df, label_col, metric=kwargs.get("metric", "euclidean"), permutations=kwargs.get("permutations", 999))

    elif test_name == "correlation_matrix":
        return correlation_matrix(df, label_col, method=kwargs.get("method", "pearson"))

    else:
        raise ValueError(f"Invalid test name provided: {test_name}")

# Anamoly Detection

## 1) Z-Score

In [15]:
def zscore_outliers(df, label_col=None, threshold=3):

    # dataframe validation
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if df.empty:
        raise ValueError("Dataset is empty.")

    df_copy = df.copy()

    # select numeric features automatically
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col is not None and label_col in feature_cols:
        feature_cols.remove(label_col)

    if len(feature_cols) == 0:
        raise ValueError("No numeric features available for Z-score analysis.")

    # check missing values
    if df[feature_cols].isnull().any().any():
        raise ValueError("Missing values detected in numeric features.")

    # feature matrix
    X = df[feature_cols]

    # compute Z-scores
    z_scores = np.abs(stats.zscore(X))
    anomaly_mask = (z_scores > threshold).any(axis=1)

    anomalies = df.loc[anomaly_mask].copy()

    # add label column (1 = anomaly, 0 = normal)
    df_copy["anomaly_label"] = anomaly_mask.astype(int)

    return df_copy

## 2) Isolation Forest

In [12]:
def predict_isolation_forest(df, label_col=None):

    model_path = "/content/drive/MyDrive/RepoGo_Rasp/models/isolation_forest_model.pkl"

    artifact = joblib.load(model_path)

    model = artifact["model"]
    scaler = artifact["scaler"]
    feature_cols = artifact["features"]

    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be pandas DataFrame")

    df_copy = df.copy()

    # If label exists, ignore it for prediction
    if label_col is not None and label_col in df_copy.columns:
        X = df_copy.drop(columns=[label_col])
    else:
        X = df_copy

    missing_cols = [c for c in feature_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    # Keep only training features
    X = X[feature_cols]

    X_scaled = scaler.transform(X)

    # Predictions
    predictions = model.predict(X_scaled)
    scores = model.decision_function(X_scaled)

    anomaly_mask = predictions == -1
    anomalies = df.loc[anomaly_mask]

    # Convert to 0/1 label
    df_copy["anomaly_label"] = (predictions == -1).astype(int)

    result = {
        "total_rows": int(len(df)),
        "num_anomalies": int(anomaly_mask.sum()),
        "anomaly_indices": anomalies.index.tolist(),
        "anomaly_rows": df_copy.to_dict(orient="records")
    }

    return df_copy

In [13]:
predict_isolation_forest(df, label_col="Group")

,Permeability,LDL_uptake,Total_ROS,Vascular_Marker,Cell_Signalling,Tube_formation,In_vivo_recovery,Group,anomaly_label
0,74,0.30,2.2,61,47,31,33,0,0
1,20,1.20,1.0,100,80,100,70,1,0
2,73,0.20,1.7,67,45,44,32,0,0
3,22,1.20,0.8,96,79,92,72,1,0
4,79,0.19,1.8,70,43,41,28,0,0
...,...,...,...,...,...,...,...,...,...
79,12,1.80,0.6,84,91,96,71,1,0
80,87,1.80,2.2,73,44,35,33,0,1
81,56,0.14,1.5,71,52,42,17,0,0
82,24,1.30,1.1,98,79,98,83,1,0


In [17]:
def combined_anomaly_detection(df, label_col=None, z_threshold=3):

    # Run both methods
    df_z = zscore_outliers(df, label_col=label_col, threshold=z_threshold)
    df_iso = predict_isolation_forest(df)

    df_final = df.copy()

    # Add both labels
    df_final["zscore_anomaly"] = df_z["anomaly_label"]
    df_final["isoforest_anomaly"] = df_iso["anomaly_label"]

    # Overlap
    df_final["overlap_anomaly"] = (
        (df_final["zscore_anomaly"] == 1) &
        (df_final["isoforest_anomaly"] == 1)
    ).astype(int)

    # PCA
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col and label_col in feature_cols:
        feature_cols.remove(label_col)

    X = df[feature_cols]

    # scale before PCA
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)

    # Prepare Plot Data
    normal_mask = df_final["isoforest_anomaly"] == 0
    anomaly_mask = df_final["isoforest_anomaly"] == 1

    normal_points = X_pca[normal_mask.values].tolist()
    anomaly_points = X_pca[anomaly_mask.values].tolist()

    # Stats
    total_rows = len(df_final)

    z_count = int(df_final["zscore_anomaly"].sum())
    iso_count = int(df_final["isoforest_anomaly"].sum())
    overlap_count = int(df_final["overlap_anomaly"].sum())

    stats = {
        "total_rows": total_rows,
        "zscore_anomalies": z_count,
        "isolation_forest_anomalies": iso_count,
        "overlap_anomalies": overlap_count,
        "zscore_percentage": round(z_count / total_rows * 100, 2),
        "isoforest_percentage": round(iso_count / total_rows * 100, 2),
        "overlap_percentage": round(overlap_count / total_rows * 100, 2)
    }

    return {
        "stats": stats,
          "plot_data": {
            "normal": normal_points,
            "isolation_forest_anomaly": anomaly_points
        },
        "data": df_final.to_dict(orient="records")
    }

In [18]:
combined_anomaly_detection(df, label_col="Group")

{'stats': {'total_rows': 84,
  'zscore_anomalies': 0,
  'isolation_forest_anomalies': 5,
  'overlap_anomalies': 0,
  'zscore_percentage': 0.0,
  'isoforest_percentage': 5.95,
  'overlap_percentage': 0.0},
 'plot_data': {'normal': [[-2.865060610646162, 0.18690263774419805],
   [2.2823661997770794, -0.6529289212834583],
   [-2.327490887183577, -0.19363266799107368],
   [2.1545857698972735, -0.47490760628551637],
   [-2.5340799145757282, -0.24648961680365747],
   [-2.17651593380644, -0.2598667349351056],
   [-2.2185166598548762, -0.5806639784746923],
   [3.260930030783408, 0.9173067288665117],
   [2.873626477639823, 0.3766511436450176],
   [-2.7689920903545095, -0.13822876257387284],
   [2.4682890960512145, 0.2846125181857753],
   [2.5172381978767464, -0.49043240139427713],
   [1.7906660437342878, 0.21880703488893938],
   [-2.7542986599915205, -0.04115812379442191],
   [-2.41699685280203, -0.13488713601032898],
   [2.5678833855610845, -1.3457775835590582],
   [-2.5704117416598637, 0.13976

# Predictive Modelling

## Classification

In [ ]:
def validate_input(df, label_col=None):

    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if df.empty:
        raise ValueError("Dataset is empty.")

    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col is not None:
        if label_col not in df.columns:
            raise ValueError(f"{label_col} not found in dataset.")

        if label_col in feature_cols:
            feature_cols.remove(label_col)

        if df[label_col].nunique() != 2:
            raise ValueError("Binary classification requires exactly 2 classes.")

    if len(feature_cols) == 0:
        raise ValueError("No numeric features available for Logistic Regression.")

    if df[feature_cols].isnull().any().any():
        raise ValueError("Missing values detected in numeric features.")

    if label_col is not None:
        if len(df) < 10:
            raise ValueError("Dataset too small for modeling.")

### Multicollinearity

In [ ]:
def compute_vif(df, label_col):

    # Validation
    validate_input(df, label_col)

    X = df.drop(columns=[label_col])
    X = X.select_dtypes(include=[np.number])

    if X.shape[1] < 2:
        raise ValueError("VIF requires at least two numeric features.")

    vif_values = []

    for i in range(X.shape[1]):
        vif_score = float(variance_inflation_factor(X.values, i))
        vif_values.append({
            "feature": X.columns[i],
            "value": round(vif_score, 2)
        })

    # Sort descending
    vif_values = sorted(vif_values, key=lambda x: x["value"], reverse=True)

    return {
        "vif": vif_values
    }

In [ ]:
compute_vif(df, label_col="Group")

{'vif': [{'feature': 'Tube_formation', 'value': 172.32},
  {'feature': 'Cell_Signalling', 'value': 154.3},
  {'feature': 'Vascular_Marker', 'value': 150.12},
  {'feature': 'In_vivo_recovery', 'value': 68.61},
  {'feature': 'Permeability', 'value': 30.94},
  {'feature': 'Total_ROS', 'value': 27.91},
  {'feature': 'LDL_uptake', 'value': 8.36}]}

## Logistic Regression

In [ ]:
def predict_logistic(df, label_col=None, model_path=None, threshold=0.5):
    validate_input(df, label_col)

    # Validate threshold
    if not (0 <= threshold <= 1):
        raise ValueError("Threshold must be between 0 and 1")

    # Load trained model
    artifact = joblib.load(model_path)

    model = artifact["model"]
    scaler = artifact["scaler"]
    feature_cols = artifact["features"]

    # Feature validation
    missing_cols = [c for c in feature_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required features: {missing_cols}")

    # Prepare X
    X = df[feature_cols].copy()
    X_scaled = scaler.transform(X)

    probs = model.predict_proba(X_scaled)[:, 1]
    preds = (probs >= threshold).astype(int)

    # Probabilities + Predictions
    prediction_table = X.copy()
    prediction_table["predicted"] = preds
    prediction_table["probability"] = probs

    # Predicted label-wise describe (frontend-ready)
    predicted_describe = {}
    for label, group in prediction_table.groupby("predicted"):
        predicted_describe[int(label)] = (
            group[feature_cols].describe().round(4).to_dict()
        )

    result = {
        "num_samples": int(len(df)),
        "num_features": int(len(feature_cols)),
        "predictions": preds.tolist(),
        "probabilities": probs.tolist(),
        "threshold_used": float(threshold),
        "prediction_table": prediction_table.to_dict(orient="records"),
        "predicted_label_describe": predicted_describe
    }

    if label_col is not None and label_col in df.columns:
        y_true = df[label_col]

        accuracy = accuracy_score(y_true, preds)
        auc_score = roc_auc_score(y_true, probs)

        conf_matrix = confusion_matrix(y_true, preds)
        class_report = classification_report(y_true, preds, output_dict=True)

        fpr, tpr, _ = roc_curve(y_true, probs)

        prediction_table["actual"] = y_true.values

        result.update({
            "data_summary": {
                "class_distribution": y_true.value_counts().to_dict()
            },
            "accuracy": float(accuracy),
            "auc_score": float(auc_score),
            "confusion_matrix": conf_matrix.tolist(),
            "classification_report": class_report,
            "roc_curve": {
                "fpr": fpr.tolist(),
                "tpr": tpr.tolist()
            }
        })

    return result

In [ ]:
predict_logistic(df, label_col="Group", model_path="/content/drive/MyDrive/RepoGo_Rasp/models/logistic_model.pkl", threshold=0.5)

{'num_samples': 84,
 'num_features': 7,
 'predictions': [0,
  1,
  0,
  1,
  0,
  0,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  0,
  0,
  0,
  1,
  0,
  1,
  0,
  0,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  0,
  0,
  0,
  1,
  0,
  1,
  0,
  0,
  0,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  0,
  0,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  0,
  0,
  1,
  0,
  0,
  1,
  0],
 'probabilities': [4.470946254711647e-06,
  0.9999634969465899,
  4.4436538243726186e-05,
  0.9999355883917934,
  1.4102100778781353e-05,
  6.0073157427292426e-05,
  4.113012451104407e-05,
  0.9999990107760715,
  0.9999969385108718,
  6.781409542956684e-06,
  0.9999872877629844,
  0.9999923479383153,
  0.9998207047326435,
  5.990832278818843e-06,
  1.7653357811075855e-05,
  0.00012440195435173696,
  0.9999937148499861,
  1.6460386410440983e-05,
  0.9999516720647771,
  6.321130767916119e-06,
  7.55217870748091e-05,
  0.

## Health Score Creation

## 1). Equal Weighting (EW) Method

In [ ]:
def predict_health_score_EW(df, model_path, label_col=None):

    # load saved artifact
    artifact = joblib.load(model_path)

    model = artifact["model"]
    scaler = artifact["scaler"]
    feature_cols = artifact["features"]

    # ensure required features exist
    missing_cols = [col for col in feature_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns: {missing_cols}")

    # select features only
    X = df[feature_cols].copy()

    # scale using saved scaler
    X_scaled = scaler.transform(X)

    # predict
    predictions = model.predict(X_scaled)

    # build JSON-friendly output
    data = df.copy()
    data["Predicted_Health_Score_EW"] = predictions

    return {
        "predictions": data.to_dict(orient="records")
    }

In [ ]:
predict_health_score_EW(df, model_path="/content/drive/MyDrive/RepoGo_Rasp/models/Health_Score_EW.pkl", label_col="Group")

{'predictions': [{'Permeability': 74,
   'LDL_uptake': 0.3,
   'Total_ROS': 2.2,
   'Vascular_Marker': 61,
   'Cell_Signalling': 47,
   'Tube_formation': 31,
   'In_vivo_recovery': 33,
   'Group': 0,
   'Predicted_Health_Score_EW': 14.310353337628662},
  {'Permeability': 20,
   'LDL_uptake': 1.2,
   'Total_ROS': 1.0,
   'Vascular_Marker': 100,
   'Cell_Signalling': 80,
   'Tube_formation': 100,
   'In_vivo_recovery': 70,
   'Group': 1,
   'Predicted_Health_Score_EW': 79.9448236772568},
  {'Permeability': 73,
   'LDL_uptake': 0.2,
   'Total_ROS': 1.7,
   'Vascular_Marker': 67,
   'Cell_Signalling': 45,
   'Tube_formation': 44,
   'In_vivo_recovery': 32,
   'Group': 0,
   'Predicted_Health_Score_EW': 21.088760615428384},
  {'Permeability': 22,
   'LDL_uptake': 1.2,
   'Total_ROS': 0.8,
   'Vascular_Marker': 96,
   'Cell_Signalling': 79,
   'Tube_formation': 92,
   'In_vivo_recovery': 72,
   'Group': 1,
   'Predicted_Health_Score_EW': 77.95798739365047},
  {'Permeability': 79,
   'LDL_upt

## 2). UnEqual Weighting (UEW) Method

In [ ]:
def compute_health_score_UEW(df, wt_LDL=0.16, wt_VM=0.12, wt_CS=0.16, wt_TF=0.12, wt_IVR=0.12, wt_PB=0.16, wt_ROS=0.16):
    df_copy = df.copy()

    weights = {
    "LDL_uptake": wt_LDL,
    "Vascular_Marker": wt_VM,
    "Cell_Signalling": wt_CS,
    "Tube_formation": wt_TF,
    "In_vivo_recovery": wt_IVR,
    "Permeability": wt_PB,
    "Total_ROS": wt_ROS
    }

    # Indicators where higher = better
    positive_indicators = ["LDL_uptake", "Vascular_Marker", "Cell_Signalling", "Tube_formation","In_vivo_recovery"]

    # Indicators where higher = worse (need inversion)
    negative_indicators = ["Permeability", "Total_ROS"]

    # All indicators
    all_indicators = positive_indicators + negative_indicators

    # validation
    if set(weights.keys()) != set(all_indicators):
        raise ValueError("Weights must be provided for all indicators")

    if abs(sum(weights.values()) - 1.0) > 1e-6:
        raise ValueError("Weights must sum to 1")

    # Normalize using SAME formula everywhere
    df_norm = df[all_indicators].copy()

    # Normalize positive indicators
    for col in positive_indicators:
        min_val = df[col].min()
        max_val = df[col].max()

        if max_val == min_val:
            df_norm[col] = 0  # avoid division by zero
        else:
            df_norm[col] = (df[col] - min_val) / (max_val - min_val)

    # Normalize negative indicators (invert)
    for col in negative_indicators:
        min_val = df[col].min()
        max_val = df[col].max()

        if max_val == min_val:
            df_norm[col] = 0
        else:
            df_norm[col] = (max_val - df[col]) / (max_val - min_val)

    # Health Score (Equal weight)
    predictions = sum(
        100 * df_norm[col] * weights[col] for col in all_indicators
    )

    # build JSON-friendly output
    data = df.copy()
    data["Predicted_Health_Score_UEW"] = predictions

    return {
        "predictions": data.to_dict(orient="records")
    }

In [ ]:
compute_health_score_UEW(df, wt_LDL=0.16, wt_VM=0.12, wt_CS=0.16, wt_TF=0.12, wt_IVR=0.12, wt_PB=0.16, wt_ROS=0.16)

{'predictions': [{'Permeability': 74,
   'LDL_uptake': 0.3,
   'Total_ROS': 2.2,
   'Vascular_Marker': 61,
   'Cell_Signalling': 47,
   'Tube_formation': 31,
   'In_vivo_recovery': 33,
   'Group': 0,
   'Predicted_Health_Score_UEW': 14.874727308051346},
  {'Permeability': 20,
   'LDL_uptake': 1.2,
   'Total_ROS': 1.0,
   'Vascular_Marker': 100,
   'Cell_Signalling': 80,
   'Tube_formation': 100,
   'In_vivo_recovery': 70,
   'Group': 1,
   'Predicted_Health_Score_UEW': 78.49594899740086},
  {'Permeability': 73,
   'LDL_uptake': 0.2,
   'Total_ROS': 1.7,
   'Vascular_Marker': 67,
   'Cell_Signalling': 45,
   'Tube_formation': 44,
   'In_vivo_recovery': 32,
   'Group': 0,
   'Predicted_Health_Score_UEW': 21.183892445291722},
  {'Permeability': 22,
   'LDL_uptake': 1.2,
   'Total_ROS': 0.8,
   'Vascular_Marker': 96,
   'Cell_Signalling': 79,
   'Tube_formation': 92,
   'In_vivo_recovery': 72,
   'Group': 1,
   'Predicted_Health_Score_UEW': 77.01202832180478},
  {'Permeability': 79,
   'LD

In [ ]:
def combine_health_scores(df, model_path, label_col=None, **uew_weights):

    # Run both methods
    ew_result = predict_health_score_EW(df, model_path, label_col)
    uew_result = compute_health_score_UEW(df, **uew_weights)

    # Convert JSON → DataFrame
    df_ew = pd.DataFrame(ew_result["predictions"])
    df_uew = pd.DataFrame(uew_result["predictions"])

    # Extract prediction columns
    ew_col = df_ew[["Predicted_Health_Score_EW"]]
    uew_col = df_uew[["Predicted_Health_Score_UEW"]]

    # Combine with original df
    final_df = df.copy().reset_index(drop=True)
    final_df = pd.concat([final_df, ew_col, uew_col], axis=1)

    # SUMMARY
    summary = final_df.describe().to_dict()

    # PARALLEL PLOT DATA
    parallel_plot = get_parallel_data(final_df, label_col=label_col)

    return {
        "data_EW": df_ew.to_dict(orient="records"),
        "data_UEW": df_uew.to_dict(orient="records"),
        "data_combined": final_df.to_dict(orient="records"),
        "summary": summary,
        "parallel_plot": parallel_plot
    }

In [ ]:
combine_health_scores(df, model_path="/content/drive/MyDrive/RepoGo_Rasp/models/Health_Score_EW.pkl")

{'data_EW': [{'Permeability': 74,
   'LDL_uptake': 0.3,
   'Total_ROS': 2.2,
   'Vascular_Marker': 61,
   'Cell_Signalling': 47,
   'Tube_formation': 31,
   'In_vivo_recovery': 33,
   'Group': 0,
   'Predicted_Health_Score_EW': 14.310353337628662},
  {'Permeability': 20,
   'LDL_uptake': 1.2,
   'Total_ROS': 1.0,
   'Vascular_Marker': 100,
   'Cell_Signalling': 80,
   'Tube_formation': 100,
   'In_vivo_recovery': 70,
   'Group': 1,
   'Predicted_Health_Score_EW': 79.9448236772568},
  {'Permeability': 73,
   'LDL_uptake': 0.2,
   'Total_ROS': 1.7,
   'Vascular_Marker': 67,
   'Cell_Signalling': 45,
   'Tube_formation': 44,
   'In_vivo_recovery': 32,
   'Group': 0,
   'Predicted_Health_Score_EW': 21.088760615428384},
  {'Permeability': 22,
   'LDL_uptake': 1.2,
   'Total_ROS': 0.8,
   'Vascular_Marker': 96,
   'Cell_Signalling': 79,
   'Tube_formation': 92,
   'In_vivo_recovery': 72,
   'Group': 1,
   'Predicted_Health_Score_EW': 77.95798739365047},
  {'Permeability': 79,
   'LDL_uptake'

# Clustering Analysis

In [ ]:
# Elbow Plot
def elbow_info(df, label_col=None, max_clusters=10):
    # Feature selection
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col and label_col in feature_cols:
        feature_cols.remove(label_col)

    if len(feature_cols) == 0:
        raise ValueError("No numeric features available")

    X = df[feature_cols].copy()

    if X.isnull().sum().sum() > 0:
        raise ValueError("Missing values found")

    # Scaling
    scaler = StandardScaler()
    x_scaled = scaler.fit_transform(X)

    ks, wcss, silhouettes = [], [], []

    n_samples = x_scaled.shape[0]
    max_k = min(max_clusters, n_samples // 2)

    for k in range(2, max_k + 1):
        kmeans = KMeans(n_clusters=k, init="k-means++", n_init=20, random_state=42)
        labels = kmeans.fit_predict(x_scaled)

        ks.append(k)
        wcss.append(kmeans.inertia_)

        # safe silhouette
        if len(set(labels)) > 1:
            silhouettes.append(silhouette_score(x_scaled, labels))
        else:
            silhouettes.append(-1)

    # Automatic k selection (BEST silhouette)
    default_k = ks[np.argmax(silhouettes)]

    # Convert to ECharts format
    wcss_points = [[k, w] for k, w in zip(ks, wcss)]
    silhouette_points = [[k, s] for k, s in zip(ks, silhouettes)]

    return {
        "k": ks,
        "wcss": wcss,
        "silhouette": silhouettes,
        "plot_data": {
            "wcss": wcss_points,
            "silhouette": silhouette_points
        },
        "auto_k": default_k
    }

In [ ]:
# Elbow + auto k
elbow_info(df, label_col="Group", max_clusters=10)

{'k': [2, 3, 4, 5, 6, 7, 8, 9, 10],
 'wcss': [89.15943161068293,
  71.36254324292902,
  56.57202475534631,
  48.78395445225856,
  41.90462771795674,
  37.65701244820599,
  33.2751635033787,
  30.90439327249726,
  28.47204195190869],
 'silhouette': [np.float64(0.7323005423580233),
  np.float64(0.6518860907080458),
  np.float64(0.46195145099535995),
  np.float64(0.29645865157352475),
  np.float64(0.2706761354411362),
  np.float64(0.27784523876672995),
  np.float64(0.28121666462805733),
  np.float64(0.2565991466329189),
  np.float64(0.2218380844909025)],
 'plot_data': {'wcss': [[2, 89.15943161068293],
   [3, 71.36254324292902],
   [4, 56.57202475534631],
   [5, 48.78395445225856],
   [6, 41.90462771795674],
   [7, 37.65701244820599],
   [8, 33.2751635033787],
   [9, 30.90439327249726],
   [10, 28.47204195190869]],
  'silhouette': [[2, np.float64(0.7323005423580233)],
   [3, np.float64(0.6518860907080458)],
   [4, np.float64(0.46195145099535995)],
   [5, np.float64(0.29645865157352475)],
 

In [ ]:
# Final KMeans Fitting
def fitting_K_means(x_scaled, k, y=None):

    kmeans_model = KMeans(n_clusters=k, init="k-means++", n_init=50, random_state=42)
    labels = kmeans_model.fit_predict(x_scaled)

    # Metrics
    metrics = {
        "silhouette": (
            silhouette_score(x_scaled, labels)
            if len(set(labels)) > 1 else -1
        )
    }

    if y is not None:
        metrics["ari"] = adjusted_rand_score(y, labels)

    return {
        "labels": labels.tolist(),
        "metrics": metrics
    }

In [ ]:
def K_means_model(df, label_col=None, max_clusters=10, user_k=None):
    # Feature selection
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col and label_col in feature_cols:
        feature_cols.remove(label_col)

    if len(feature_cols) == 0:
        raise ValueError("No numeric features available")

    X = df[feature_cols].copy()

    if X.isnull().sum().sum() > 0:
        raise ValueError("Missing values found")

    # Scaling
    scaler = StandardScaler()
    x_scaled = scaler.fit_transform(X)

    # Elbow + auto k
    elbow = elbow_info(df, max_clusters=max_clusters)

    # Select k
    selected_k = user_k if user_k is not None else elbow["auto_k"]

    # Final clustering
    y_true = df[label_col] if label_col and label_col in df.columns else None
    result = fitting_K_means(x_scaled, selected_k, y_true)

    labels = result["labels"]
    metrics = result["metrics"]

    # PCA (2D)
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(x_scaled)

    # FORMAT: [x, y, label]
    plot_data = [
        [float(X_pca[i, 0]), float(X_pca[i, 1]), int(labels[i])]
        for i in range(len(labels))
    ]

    # Output dataframe
    output_df = df.copy()
    output_df["cluster"] = labels

    table = output_df.to_dict(orient="records")

    return {
        "df": table,
        "features_used": feature_cols,
        "selected_k": selected_k,
        "metrics": metrics,
        "plot_data": plot_data
    }

In [ ]:
K_means_model(df, "Group", user_k=None)

{'df': [{'Permeability': 74,
   'LDL_uptake': 0.3,
   'Total_ROS': 2.2,
   'Vascular_Marker': 61,
   'Cell_Signalling': 47,
   'Tube_formation': 31,
   'In_vivo_recovery': 33,
   'Group': 0,
   'cluster': 0},
  {'Permeability': 20,
   'LDL_uptake': 1.2,
   'Total_ROS': 1.0,
   'Vascular_Marker': 100,
   'Cell_Signalling': 80,
   'Tube_formation': 100,
   'In_vivo_recovery': 70,
   'Group': 1,
   'cluster': 1},
  {'Permeability': 73,
   'LDL_uptake': 0.2,
   'Total_ROS': 1.7,
   'Vascular_Marker': 67,
   'Cell_Signalling': 45,
   'Tube_formation': 44,
   'In_vivo_recovery': 32,
   'Group': 0,
   'cluster': 0},
  {'Permeability': 22,
   'LDL_uptake': 1.2,
   'Total_ROS': 0.8,
   'Vascular_Marker': 96,
   'Cell_Signalling': 79,
   'Tube_formation': 92,
   'In_vivo_recovery': 72,
   'Group': 1,
   'cluster': 1},
  {'Permeability': 79,
   'LDL_uptake': 0.19,
   'Total_ROS': 1.8,
   'Vascular_Marker': 70,
   'Cell_Signalling': 43,
   'Tube_formation': 41,
   'In_vivo_recovery': 28,
   'Group

## 2). DBSCAN

In [ ]:
def fit_dbscan(df, label_col=None, min_samples=5):
    df = df.copy()

    # Select numeric features
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col and label_col in feature_cols:
        feature_cols.remove(label_col)

    if len(feature_cols) == 0:
        raise ValueError("No numeric features found")

    X = df[feature_cols].values
    y = df[label_col].values if label_col else None

    if np.isnan(X).any():
        raise ValueError("NaNs detected — handle before training")

    # Scaling
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Auto eps (merged here)
    neighbors = NearestNeighbors(n_neighbors=min_samples)
    neighbors_fit = neighbors.fit(X_scaled)
    distances, _ = neighbors_fit.kneighbors(X_scaled)

    k_distances = np.sort(distances[:, -1])
    eps = np.median(k_distances) * 1.2   # stable heuristic

    # Fit DBSCAN
    model = DBSCAN(eps=eps, min_samples=min_samples, metric="euclidean")
    clusters = model.fit_predict(X_scaled)

    clusters = np.array(clusters)

    # Metrics
    metrics = {}

    valid_mask = clusters != -1
    n_valid_clusters = len(set(clusters[valid_mask]))

    if n_valid_clusters > 1:
        metrics["silhouette"] = float(
            silhouette_score(X_scaled[valid_mask], clusters[valid_mask])
        )
    else:
        metrics["silhouette"] = None

    if y is not None:
        metrics["ari"] = float(adjusted_rand_score(y, clusters))
    else:
        metrics["ari"] = None

    return X_scaled, clusters, metrics, eps, feature_cols

In [ ]:
def DBSCAN_predict(df, label_col=None, min_samples=5):

    np.random.seed(42)

    # Step 1+2 merged
    X_scaled, clusters, metrics, eps, feature_cols = fit_dbscan(df, label_col, min_samples)

    # Cluster stats
    n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
    noise_points = int(np.sum(clusters == -1))
    noise_ratio = noise_points / len(clusters)

    # Safety checks
    if n_clusters < 2:
        raise ValueError("DBSCAN found <2 clusters. Adjust parameters.")

    if noise_ratio > 0.6:
        raise ValueError(f"Too many noise points: {noise_ratio:.2f}")

    # PCA for visualization
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)

    # FORMAT: [x, y, label]
    plot_data = [
        [float(X_pca[i, 0]), float(X_pca[i, 1]), int(clusters[i])]
        for i in range(len(clusters))
    ]

    # Attach clusters to df
    df_out = df.copy()
    df_out["cluster"] = clusters

    return {
        "eps": round(float(eps), 4),
        "min_samples": min_samples,
        "n_clusters": int(n_clusters),
        "noise_points": noise_points,
        "noise_ratio": round(float(noise_ratio), 4),
        "features": feature_cols,
        "metrics": metrics,
        #"clusters": clusters.tolist(),
        "df": df_out.to_dict(orient="records"),
        "plot_data": plot_data,
    }

In [ ]:
DBSCAN_predict(df, label_col="Group", min_samples=5)

{'eps': 0.8471,
 'min_samples': 5,
 'n_clusters': 3,
 'noise_points': 8,
 'noise_ratio': 0.0952,
 'features': ['Permeability',
  'LDL_uptake',
  'Total_ROS',
  'Vascular_Marker',
  'Cell_Signalling',
  'Tube_formation',
  'In_vivo_recovery'],
 'metrics': {'silhouette': 0.49998996436307747, 'ari': 0.703495349014657},
 'df': [{'Permeability': 74,
   'LDL_uptake': 0.3,
   'Total_ROS': 2.2,
   'Vascular_Marker': 61,
   'Cell_Signalling': 47,
   'Tube_formation': 31,
   'In_vivo_recovery': 33,
   'Group': 0,
   'cluster': 0},
  {'Permeability': 20,
   'LDL_uptake': 1.2,
   'Total_ROS': 1.0,
   'Vascular_Marker': 100,
   'Cell_Signalling': 80,
   'Tube_formation': 100,
   'In_vivo_recovery': 70,
   'Group': 1,
   'cluster': 1},
  {'Permeability': 73,
   'LDL_uptake': 0.2,
   'Total_ROS': 1.7,
   'Vascular_Marker': 67,
   'Cell_Signalling': 45,
   'Tube_formation': 44,
   'In_vivo_recovery': 32,
   'Group': 0,
   'cluster': 0},
  {'Permeability': 22,
   'LDL_uptake': 1.2,
   'Total_ROS': 0.8,

In [ ]:
def combined_clustering_analysis(df, label_col=None, max_clusters=10, user_k=None, min_samples=5):

    # Run models
    kmeans_res = K_means_model(df, label_col, max_clusters, user_k)
    dbscan_res = DBSCAN_predict(df, label_col, min_samples)

    df_k = pd.DataFrame(kmeans_res["df"])
    df_d = pd.DataFrame(dbscan_res["df"])

    # Base dataframe
    df_final = df.copy()

    # Add cluster columns
    df_final["kmeans_cluster"] = df_k["cluster"].values
    df_final["dbscan_cluster"] = df_d["cluster"].values

    # keep label_col if exists
    if label_col and label_col in df.columns:
        df_final[label_col] = df[label_col]

    # Summary Function
    def get_summary(labels):
        summary = (
            pd.Series(labels)
            .value_counts()
            .reset_index()
        )
        summary.columns = ["cluster", "count"]
        summary = summary.sort_values(by="count", ascending=True).reset_index(drop=True)
        return summary.to_dict(orient="records")

    # Summaries
    kmeans_summary = get_summary(df_final["kmeans_cluster"])
    dbscan_summary = get_summary(df_final["dbscan_cluster"])

    # Final Return
    return {
        "data": df_final.to_dict(orient="records"),
        "kmeans_summary": kmeans_summary,
        "dbscan_summary": dbscan_summary,
        "kmeans_k": kmeans_res["selected_k"],
        "dbscan_eps": dbscan_res["eps"]
    }

In [ ]:
combined_clustering_analysis(df, label_col="Group", max_clusters=10, user_k=None, min_samples=5)

{'data': [{'Permeability': 74,
   'LDL_uptake': 0.3,
   'Total_ROS': 2.2,
   'Vascular_Marker': 61,
   'Cell_Signalling': 47,
   'Tube_formation': 31,
   'In_vivo_recovery': 33,
   'Group': 0,
   'kmeans_cluster': 0,
   'dbscan_cluster': 0},
  {'Permeability': 20,
   'LDL_uptake': 1.2,
   'Total_ROS': 1.0,
   'Vascular_Marker': 100,
   'Cell_Signalling': 80,
   'Tube_formation': 100,
   'In_vivo_recovery': 70,
   'Group': 1,
   'kmeans_cluster': 1,
   'dbscan_cluster': 1},
  {'Permeability': 73,
   'LDL_uptake': 0.2,
   'Total_ROS': 1.7,
   'Vascular_Marker': 67,
   'Cell_Signalling': 45,
   'Tube_formation': 44,
   'In_vivo_recovery': 32,
   'Group': 0,
   'kmeans_cluster': 0,
   'dbscan_cluster': 0},
  {'Permeability': 22,
   'LDL_uptake': 1.2,
   'Total_ROS': 0.8,
   'Vascular_Marker': 96,
   'Cell_Signalling': 79,
   'Tube_formation': 92,
   'In_vivo_recovery': 72,
   'Group': 1,
   'kmeans_cluster': 1,
   'dbscan_cluster': 1},
  {'Permeability': 79,
   'LDL_uptake': 0.19,
   'Tota

# Vizualization

## 1). Parallel plot

In [ ]:
def get_parallel_data(df, label_col="None"):

    # Only feature columns
    feature_cols = df.select_dtypes(include="number").columns.tolist()

    # Remove label column if provided and exists
    if label_col is not None and label_col in feature_cols:
        feature_cols.remove(label_col)

    result = {
        "chart_type": "parallel",
        "dimensions": feature_cols,
        "data": df[feature_cols].values.tolist()
    }

    # Add group only if label_col is provided and exists
    if label_col is not None and label_col in df.columns:
        result["group"] = df[label_col].tolist()

    return result

In [ ]:
get_parallel_data(df, label_col="Group")

{'chart_type': 'parallel',
 'dimensions': ['Permeability',
  'LDL_uptake',
  'Total_ROS',
  'Vascular_Marker',
  'Cell_Signalling',
  'Tube_formation',
  'In_vivo_recovery'],
 'data': [[74.0, 0.3, 2.2, 61.0, 47.0, 31.0, 33.0],
  [20.0, 1.2, 1.0, 100.0, 80.0, 100.0, 70.0],
  [73.0, 0.2, 1.7, 67.0, 45.0, 44.0, 32.0],
  [22.0, 1.2, 0.8, 96.0, 79.0, 92.0, 72.0],
  [79.0, 0.19, 1.8, 70.0, 43.0, 41.0, 28.0],
  [62.0, 0.32, 1.8, 73.0, 46.0, 38.0, 24.0],
  [75.0, 0.22, 2.0, 81.0, 49.0, 32.0, 32.0],
  [22.0, 2.3, 0.7, 96.0, 94.0, 97.0, 81.0],
  [21.0, 1.8, 0.8, 92.0, 95.0, 96.0, 82.0],
  [77.0, 0.25, 2.4, 68.0, 42.0, 44.0, 33.0],
  [16.0, 1.5, 0.5, 83.0, 89.0, 94.0, 75.0],
  [13.0, 1.1, 0.6, 89.0, 89.0, 99.0, 77.0],
  [22.0, 1.3, 0.9, 78.0, 82.0, 99.0, 71.0],
  [74.0, 0.27, 2.1, 67.0, 42.0, 32.0, 32.0],
  [69.0, 0.41, 2.3, 74.0, 55.0, 34.0, 21.0],
  [89.0, 2.0, 1.7, 70.0, 47.0, 37.0, 29.0],
  [12.0, 0.7, 0.6, 99.0, 90.0, 95.0, 77.0],
  [65.0, 0.34, 2.1, 61.0, 50.0, 47.0, 20.0],
  [26.0, 1.3, 0.

## 2). PCA Plot

In [ ]:
def pca_plot(df, label_col="Group", n_components=3):
    if n_components not in [2, 3]:
        raise ValueError("n_components must be 2 or 3")

    # Separate features and label
    X = df.drop(columns=[label_col])
    y = df[label_col]

    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Apply PCA
    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_scaled)

    # Explained variance in percentage
    explained = (pca.explained_variance_ratio_ * 100).tolist()

    # Column-wise PCA coordinates for plotting
    points = {
        "pc1": X_pca[:, 0].astype(float).tolist(),
        "pc2": X_pca[:, 1].astype(float).tolist(),
        "group": y.astype(int).tolist()
    }

    if n_components == 3:
        points["pc3"] = X_pca[:, 2].astype(float).tolist()

    # Feature loadings for biplot
    loadings = []
    for i, feature in enumerate(X.columns):
        loading = {
            "feature": feature,
            "pc1": float(pca.components_.T[i, 0]),
            "pc2": float(pca.components_.T[i, 1])
        }
        if n_components == 3:
            loading["pc3"] = float(pca.components_.T[i, 2])
        loadings.append(loading)

    return {
        "chart_type": "pca_3d" if n_components == 3 else "pca_2d",
        "n_components": n_components,
        "explained_variance_percent": explained,
        "points": points,
        "loadings": loadings
    }

    def pca_plot(self, n_components=3):
      if n_components not in [2, 3]:
          raise ValueError("n_components must be 2 or 3")

      df, lc = self.df, self.label_col

      # Separate features and label
      X = df.drop(columns=[lc])
      y = df[lc]

      # Standardize features
      scaler = StandardScaler()
      X_scaled = scaler.fit_transform(X)

      # Apply PCA
      pca = PCA(n_components=n_components)
      X_pca = pca.fit_transform(X_scaled)

      # Explained variance in percentage
      explained = [round(float(v * 100), 2) for v in pca.explained_variance_ratio_]

      groups = y.unique().tolist()

      # Feature loadings table
      loading_rows = []
      for i, feature in enumerate(X.columns):
          row = [feature] + [round(float(pca.components_.T[i, j]), 4) for j in range(n_components)]
          loading_rows.append(row)

      loadings_table = {
          "id": "table-pca-loadings",
          "title": "PCA Feature Loadings",
          "columns": ["Feature"] + [f"PC{j+1}" for j in range(n_components)],
          "rows": loading_rows
      }

      variance_table = {
          "id": "table-pca-variance",
          "title": "PCA Explained Variance",
          "columns": ["Component", "Explained Variance (%)"],
          "rows": [
              [f"PC{i+1}", explained[i]]
              for i in range(n_components)
          ]
      }

      if n_components == 2:
          series = []
          for group in groups:
              mask = y == group
              series.append({
                  "name": str(group),
                  "type": "scatter",
                  "data": [
                      [round(float(X_pca[i, 0]), 4), round(float(X_pca[i, 1]), 4)]
                      for i in range(len(X_pca)) if mask.iloc[i]
                  ],
                  "symbolSize": 8
              })

          chart = {
              "id": "chart-pca-2d",
              "title": "PCA 2D Scatter Plot",
              "fullWidth": True,
              "option": {
                  "tooltip": {"trigger": "item"},
                  "legend": {"data": [str(g) for g in groups]},
                  "xAxis": {
                      "type": "value",
                      "name": f"PC1 ({explained[0]}%)"
                  },
                  "yAxis": {
                      "type": "value",
                      "name": f"PC2 ({explained[1]}%)"
                  },
                  "series": series
              }
          }

      else:
          series = []
          for group in groups:
              mask = y == group
              series.append({
                  "name": str(group),
                  "type": "scatter3D",
                  "data": [
                      [round(float(X_pca[i, 0]), 4), round(float(X_pca[i, 1]), 4), round(float(X_pca[i, 2]), 4)]
                      for i in range(len(X_pca)) if mask.iloc[i]
                  ],
                  "symbolSize": 6
              })

          chart = {
              "id": "chart-pca-3d",
              "title": "PCA 3D Scatter Plot",
              "fullWidth": True,
              "option": {
                  "tooltip": {},
                  "legend": {"data": [str(g) for g in groups]},
                  "xAxis3D": {"type": "value", "name": f"PC1 ({explained[0]}%)"},
                  "yAxis3D": {"type": "value", "name": f"PC2 ({explained[1]}%)"},
                  "zAxis3D": {"type": "value", "name": f"PC3 ({explained[2]}%)"},
                  "grid3D": {"viewControl": {"autoRotate": True}},
                  "series": series
              }
          }

      return {
          "tables": [variance_table, loadings_table],
          "charts": [chart]
      }

In [ ]:
pca_plot(df, label_col="Group", n_components=2)

{'chart_type': 'pca_2d',
 'n_components': 2,
 'explained_variance_percent': [86.87342246340964, 5.173567583576602],
 'points': {'pc1': [-2.865060610646162,
   2.2823661997770794,
   -2.327490887183577,
   2.1545857698972735,
   -2.5340799145757282,
   -2.17651593380644,
   -2.2185166598548762,
   3.260930030783408,
   2.873626477639823,
   -2.7689920903545095,
   2.4682890960512145,
   2.5172381978767464,
   1.7906660437342878,
   -2.7542986599915205,
   -2.41699685280203,
   -1.6662172995500488,
   2.5678833855610845,
   -2.5704117416598637,
   2.1765866435806136,
   -2.70534113756875,
   -2.1296069248892855,
   2.319093200205272,
   3.06609142311261,
   2.2995433518760393,
   -2.3376289223958486,
   -2.920954569087344,
   2.394586316750971,
   2.5833677159857387,
   2.112774342810461,
   3.0700888083318745,
   2.3880637161471623,
   -2.2638631238981293,
   2.5414515234608257,
   -2.450812855340944,
   -2.088031542706462,
   -2.2946340761930406,
   -2.5450893441583173,
   2.0851422835

## 3) Scatter plot

In [ ]:
def scatter_plot(df, label_col="Group", feature_x=None, feature_y=None, feature_z=None):

    if feature_x is None or feature_y is None:
        raise ValueError("feature_x and feature_y are required")

    if feature_x not in df.columns or feature_y not in df.columns:
        raise ValueError("Selected features not found in dataframe")

    if feature_z and feature_z not in df.columns:
        raise ValueError("feature_z not found in dataframe")

    chart_type = "scatter_3d" if feature_z else "scatter_2d"

    points = {
        feature_x: df[feature_x].astype(float).tolist(),
        feature_y: df[feature_y].astype(float).tolist(),
        "group": df[label_col].astype(int).tolist()
    }

    if feature_z:
        points[feature_z] = df[feature_z].astype(float).tolist()

    return {
        "chart_type": chart_type,
        "features": {
            "x": feature_x,
            "y": feature_y,
            "z": feature_z
        },
        "points": points
    }

In [ ]:
scatter_plot(df, label_col="Group", feature_x="Permeability", feature_y="LDL_uptake", feature_z="Total_ROS")

{'chart_type': 'scatter_3d',
 'features': {'x': 'Permeability', 'y': 'LDL_uptake', 'z': 'Total_ROS'},
 'points': {'Permeability': [74.0,
   20.0,
   73.0,
   22.0,
   79.0,
   62.0,
   75.0,
   22.0,
   21.0,
   77.0,
   16.0,
   13.0,
   22.0,
   74.0,
   69.0,
   89.0,
   12.0,
   65.0,
   26.0,
   80.0,
   68.0,
   24.0,
   19.0,
   19.0,
   71.0,
   61.0,
   17.0,
   22.0,
   28.0,
   18.0,
   20.0,
   63.0,
   25.0,
   78.0,
   67.0,
   66.0,
   63.0,
   19.0,
   77.0,
   29.0,
   78.0,
   70.0,
   90.0,
   27.0,
   27.0,
   21.0,
   69.0,
   72.0,
   13.0,
   26.0,
   73.0,
   75.0,
   71.0,
   19.0,
   67.0,
   19.0,
   15.0,
   12.0,
   19.0,
   11.0,
   28.0,
   28.0,
   77.0,
   76.0,
   60.0,
   25.0,
   85.0,
   70.0,
   67.0,
   72.0,
   29.0,
   24.0,
   23.0,
   77.0,
   24.0,
   22.0,
   21.0,
   73.0,
   65.0,
   12.0,
   87.0,
   56.0,
   24.0,
   66.0],
  'LDL_uptake': [0.3,
   1.2,
   0.2,
   1.2,
   0.19,
   0.32,
   0.22,
   2.3,
   1.8,
   0.25,
   1.5,
   1.1,
 